In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy.stats import gaussian_kde, chi2_contingency, kruskal

sys.path.append('/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/spikeparam/AP_empirical_paper1/datasets/spe-1/spe1_helper_modules/')
import config

CLUSTER_PKL_DIR = '/Users/blancamartin/Desktop/Voytek_Lab/spike_waveform/spe1_pickles/cluster_pickles/'

CLUST_COLORS = {'low': '#0072B2', 'high': '#D55E00'}
GROUP_ORDER  = ['low', 'high']

EXAMPLE_CELL = 32

sns.set_style('ticks')
plt.rcParams.update({
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.dpi': 150,
})

# Waveform clusters are independent of ISI-based clusters

Waveform clustering uses AP shape features only — log ISI is not a clustering feature. For c32, the waveform cluster assignment (based on `peak_amp_cluster`) clearly separates waveform shape features but does not predict ISI cluster membership, and vice versa: ISI cluster assignment does not predict waveform cluster membership. The population panel shows Cramér's V (waveform cluster vs ISI quantile group) is low across all cells.

In [ ]:
# ── Helpers ───────────────────────────────────────────────────────────────────
def cramers_v(x, y):
    """Cramér's V between two categorical series."""
    ct = pd.crosstab(x, y)
    chi2, _, _, _ = chi2_contingency(ct)
    n = ct.sum().sum()
    r, c = ct.shape
    k = min(r, c)
    if k < 2 or n == 0:
        return np.nan
    return np.sqrt(chi2 / (n * (k - 1)))


def kde_panel(ax, df, feature, cluster_col, xlabel, show_legend=False):
    sub = df.dropna(subset=[feature, cluster_col])
    xs  = np.linspace(sub[feature].min(), sub[feature].max(), 400)
    for grp in GROUP_ORDER:
        vals = sub.loc[sub[cluster_col] == grp, feature].values
        if len(vals) < 5:
            continue
        kde = gaussian_kde(vals, bw_method='silverman')
        ax.plot(xs, kde(xs), color=CLUST_COLORS[grp], lw=2,
                label=f'{grp.capitalize()} (n={len(vals)})')
        ax.fill_between(xs, kde(xs), alpha=0.15, color=CLUST_COLORS[grp])
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Density')
    if show_legend:
        ax.legend(frameon=False, fontsize=9)
    sns.despine(ax=ax)


def contingency_bars(ax, df, row_col, col_col, row_order, col_colors, xlabel, title):
    """
    Grouped bar chart: for each category in row_col, show the fraction of spikes
    in each category of col_col. Bars within each group are stacked or adjacent.
    Uses adjacent bars (one bar per col_col category per row_col level).
    """
    sub   = df.dropna(subset=[row_col, col_col])
    ct    = pd.crosstab(sub[row_col], sub[col_col])
    props = ct.div(ct.sum(axis=1), axis=0)  # row-normalize → P(col | row)
    props = props.reindex(index=row_order, fill_value=0)

    n_rows = len(props)
    n_cols = len(props.columns)
    width  = 0.7 / n_cols
    x      = np.arange(n_rows)

    col_list = [c for c in row_order if c in props.columns] + \
               [c for c in props.columns if c not in row_order]

    for i, col_cat in enumerate(col_list):
        if col_cat not in props.columns:
            continue
        offset = (i - (n_cols - 1) / 2) * width
        ax.bar(x + offset, props[col_cat],
               width=width * 0.9,
               color=col_colors.get(col_cat, '#999999'),
               label=col_cat.capitalize(),
               edgecolor='white', linewidth=0.5)

    ax.axhline(0.5, color='k', lw=0.8, ls='--', alpha=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels([r.capitalize() for r in row_order])
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Fraction of spikes')
    ax.set_ylim(0, 1)
    ax.set_title(title, fontsize=11)
    ax.legend(frameon=False, fontsize=9, title='ISI group' if 'ISI' in title or 'isi' in col_col else 'Waveform group')
    sns.despine(ax=ax)

In [ ]:
# ── Load cluster pickles ──────────────────────────────────────────────────────
NO_CLUSTER = [17, 18, 43]

cell_dfs = {}
for cnum in list(config.DICT_CELL_TYPE.keys()):
    if cnum in NO_CLUSTER:
        continue
    pkl = os.path.join(CLUSTER_PKL_DIR, f'c{cnum}_cluster_df.pkl')
    if not os.path.exists(pkl):
        continue
    df = pd.read_pickle(pkl)
    if 'log_isi' not in df.columns:
        continue
    # Accept cells with either unified groups or per-feature clusters
    has_waveform_clust = 'groups' in df.columns or 'peak_amp_cluster' in df.columns
    if not has_waveform_clust:
        continue
    cell_dfs[cnum] = df

print(f'Cells loaded: {len(cell_dfs)}')

# Verify c32 structure
df32 = cell_dfs[EXAMPLE_CELL]
clust_cols = [c for c in df32.columns if 'cluster' in c or c == 'groups']
print(f'c32 cluster columns: {clust_cols}')
print(f'peak_amp_cluster values: {df32["peak_amp_cluster"].unique()}')
print(f'log_isi_cluster values:  {df32["log_isi_cluster"].unique()}')

In [ ]:
# ── Global style (match supp_cluster_distributions) ──────────────────────────
sns.set_theme(style='ticks', font_scale=1.4, rc={
    'axes.linewidth':    2.0,
    'xtick.major.width': 2.0,
    'ytick.major.width': 2.0,
    'xtick.major.size':  6,
    'ytick.major.size':  6,
    'patch.linewidth':   2.0,
    'lines.linewidth':   2.0,
})

ISI_BG = '#dddddd'   # grey background for ISI panels (matches supp_cluster_distributions)


def cramers_v(x, y):
    ct = pd.crosstab(x, y)
    chi2, _, _, _ = chi2_contingency(ct)
    n = ct.sum().sum()
    r, c = ct.shape
    k = min(r, c)
    if k < 2 or n == 0:
        return np.nan
    return np.sqrt(chi2 / (n * (k - 1)))


def kde_panel(ax, df, feature, cluster_col, group_order, colors, is_isi=False,
              clip_pct=2, min_bw_frac=0.05):
    """KDE panel styled to match supp_cluster_distributions:
    solid fill (alpha=1), thin outline (lw=0.6), no axes, grey bg for ISI.
    """
    sub = df.dropna(subset=[feature, cluster_col])
    if sub.empty:
        return
    vals_all = sub[feature]
    lo = np.percentile(vals_all, clip_pct)
    hi = np.percentile(vals_all, 100 - clip_pct)
    pad    = (hi - lo) * 0.08
    x_grid = np.linspace(lo - pad, hi + pad, 300)
    min_abs_bw = (hi - lo) * min_bw_frac

    for grp in group_order:
        vals = sub.loc[sub[cluster_col] == grp, feature].values
        if len(vals) < 5:
            continue
        std_v = vals.std()
        bw    = max(0.3, min_abs_bw / max(std_v, 1e-10))
        y     = gaussian_kde(vals, bw_method=bw)(x_grid)
        col   = colors.get(grp, '#888')
        ax.fill_between(x_grid, y, color=col, alpha=1.0)
        ax.plot(x_grid, y, color=col, lw=0.6)

    ax.set_xticks([])
    ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)
    ax.set_facecolor(ISI_BG if is_isi else 'white')

In [ ]:
# ── Population Cramér's V ─────────────────────────────────────────────────────
results = []
for cnum, df in cell_dfs.items():
    sub = df.dropna(subset=['log_isi'])
    if 'groups' in sub.columns:
        wcol = 'groups'
    elif 'peak_amp_cluster' in sub.columns:
        wcol = 'peak_amp_cluster'
    else:
        continue
    sub = sub.dropna(subset=[wcol])
    n_grps = sub[wcol].nunique()
    if len(sub) < 20 or n_grps < 2:
        continue
    sub = sub.copy()
    sub['isi_qgroup'] = pd.qcut(sub['log_isi'], q=n_grps, labels=False, duplicates='drop')
    sub = sub.dropna(subset=['isi_qgroup'])
    if sub['isi_qgroup'].nunique() < 2:
        continue
    v = cramers_v(sub[wcol], sub['isi_qgroup'])
    results.append({'cell': cnum, 'cramers_v': v, 'n': len(sub)})

df_v = pd.DataFrame(results).sort_values('cell').reset_index(drop=True)
print(f'n cells: {len(df_v)}   Median V = {df_v.cramers_v.median():.3f}')
print(f'c{EXAMPLE_CELL} V = {df_v.loc[df_v.cell == EXAMPLE_CELL, "cramers_v"].values[0]:.3f}')

# ── Figure ─────────────────────────────────────────────────────────────────────
FEATS = [
    ('peak_amp',       'Peak amplitude',          False),
    ('peak_sharpness', 'Peak sharpness',           False),
    ('exp_const',      'Repolarization constant',  False),
    ('log_isi',        'Log ISI',                  True),
]

df_ex  = cell_dfs[EXAMPLE_CELL]
wclust = 'peak_amp_cluster'
iclust = 'log_isi_cluster'

fig = plt.figure(figsize=(16, 11))
gs  = gridspec.GridSpec(3, 4, figure=fig, hspace=0.55, wspace=0.15,
                        height_ratios=[1, 1, 1])

# ── Row 1: colored by waveform cluster ───────────────────────────────────────
for col, (feat, label, is_isi) in enumerate(FEATS):
    clust = wclust if wclust in df_ex.columns else 'groups'
    ax = fig.add_subplot(gs[0, col])
    kde_panel(ax, df_ex, feat, clust, GROUP_ORDER, CLUST_COLORS, is_isi=is_isi)
    ax.set_title(label, fontsize=13, fontweight='bold', pad=6)

fig.text(0.005, 0.82, 'Colored by\nwaveform cluster', va='center',
         rotation=90, fontsize=11, color='#444', ha='center')

# ── Row 2: colored by ISI cluster ────────────────────────────────────────────
for col, (feat, label, is_isi) in enumerate(FEATS):
    ax = fig.add_subplot(gs[1, col])
    kde_panel(ax, df_ex, feat, iclust, GROUP_ORDER, CLUST_COLORS, is_isi=is_isi)

fig.text(0.005, 0.53, 'Colored by\nISI cluster', va='center',
         rotation=90, fontsize=11, color='#444', ha='center')

fig.text(0.5, 0.97, f'c{EXAMPLE_CELL}', ha='center', fontsize=14, fontweight='bold')

# Shared legend (low / high)
from matplotlib.patches import Patch
legend_handles = [Patch(facecolor=CLUST_COLORS['low'],  label='Low'),
                  Patch(facecolor=CLUST_COLORS['high'], label='High')]
fig.legend(handles=legend_handles, loc='upper right', bbox_to_anchor=(0.99, 0.97),
           frameon=False, fontsize=11, ncol=2)

# ── Row 3: population Cramér's V ─────────────────────────────────────────────
ax_pop = fig.add_subplot(gs[2, 1:3])
v_vals = df_v['cramers_v'].dropna().values
ax_pop.hist(v_vals, bins=12, color='#56B4E9', edgecolor='white', linewidth=0.5)
ax_pop.axvline(np.median(v_vals), color='k', lw=1.5, ls='--',
               label=f'Median = {np.median(v_vals):.3f}')
ex_v = df_v.loc[df_v.cell == EXAMPLE_CELL, 'cramers_v'].values[0]
ax_pop.axvline(ex_v, color='#D55E00', lw=1.5, ls=':',
               label=f'c{EXAMPLE_CELL} = {ex_v:.3f}')
ax_pop.set_xlabel("Cramér's V (waveform cluster vs ISI group)")
ax_pop.set_ylabel('Cells')
ax_pop.set_title(f'Population (n={len(df_v)} cells)', fontsize=13, fontweight='bold')
ax_pop.legend(frameon=False)
sns.despine(ax=ax_pop)

plt.show()